# 05. 시장 세분화 — 자치구 K-Means 군집 분석

**분석 목적:** 25개 자치구를 RevPAR 성과 특성에 따라 4개 군집으로 분류합니다.

**핵심 질문:**
- 서울 25개 자치구는 RevPAR 관점에서 어떤 유형으로 나뉘는가?
- 군집별 공통 특성은 무엇이며, 어떤 전략이 적합한가?

**접근 방법:**
- K-Means(k=4) 군집화 (Elbow + Silhouette으로 최적 k 결정)
- 군집 피처: median_revpar_ao, total_listings, dormant_ratio, superhost_rate

In [ ]:
# KPI_REVPAR_ACTIVE_MEDIAN은 클러스터별 성과 비교의 기준선으로 사용
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import silhouette_score
import warnings; warnings.filterwarnings("ignore")

DOMAIN_KOREAN = "서울 에어비앤비"
RANDOM_SEED = 42
REPORTS_DIR = Path("../reports")
PROCESSED_DIR = Path("../data/processed")

KPI_REVPAR_ACTIVE_MEDIAN = 47850
N_CLUSTERS = 4

sns.set_style("whitegrid")
import platform
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
else:
    plt.rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams["figure.dpi"] = 100

try:
    district_agg = pd.read_csv(PROCESSED_DIR / "district_aggregated.csv")
except FileNotFoundError:
    df = pd.read_csv("../data/processed/seoul_airbnb_features.csv")
    district_agg = df.groupby("district").agg(
        total_listings=("id","count"),
        ao_count=("is_active_operating","sum"),
        median_revpar_ao=("ttm_revpar", lambda x: x[df.loc[x.index,"is_active_operating"]==1].median() if (df.loc[x.index,"is_active_operating"]==1).any() else 0),
        dormant_count=("refined_status", lambda x: x.isin(["Dormant","Ghost"]).sum()),
        superhost_rate=("superhost","mean"), median_pop=("ttm_pop","median")
    ).reset_index()
    district_agg["dormant_ratio"] = district_agg["dormant_count"]/district_agg["total_listings"]
    district_agg["supply_share"] = district_agg["total_listings"]/district_agg["total_listings"].sum()
    district_agg.to_csv(PROCESSED_DIR/"district_aggregated.csv", index=False, encoding="utf-8-sig")

## 1️⃣ 최적 k 결정 (Elbow + Silhouette)

In [ ]:
# Elbow와 Silhouette 동시 확인 -- 엘보우만으로는 최적 k를 명확히 결정하기 어려움
cluster_features = ["median_revpar_ao","total_listings","dormant_ratio","superhost_rate"]
X_cluster = district_agg[cluster_features].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

inertias, silhouettes = [], []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(2,8), inertias, "bo-", markersize=8)
axes[0].axvline(N_CLUSTERS, color="red", linestyle="--", label=f"k={N_CLUSTERS} 선택")
axes[0].set_title("Elbow Method", fontsize=12, fontweight="bold"); axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia"); axes[0].legend()
axes[1].plot(range(2,8), silhouettes, "ro-", markersize=8)
axes[1].axvline(N_CLUSTERS, color="blue", linestyle="--", label=f"k={N_CLUSTERS} 선택")
axes[1].set_title("Silhouette Score", fontsize=12, fontweight="bold"); axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette"); axes[1].legend()
plt.tight_layout()
plt.show()
print(f"k={N_CLUSTERS} Silhouette: {silhouettes[N_CLUSTERS-2]:.4f}")

## 2️⃣ K-Means 클러스터링 (k=4)

In [ ]:
# k=4 선택 -- Elbow/Silhouette 분석과 전략적 해석 가능성을 동시에 고려한 결정
km_final = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
district_agg["cluster"] = km_final.fit_predict(X_scaled)

# RevPAR 기준으로 클러스터 순위 부여
cluster_revpar = district_agg.groupby("cluster")["median_revpar_ao"].median().sort_values(ascending=False)
rank_map = {old: new for new, old in enumerate(cluster_revpar.index)}
district_agg["cluster_rank"] = district_agg["cluster"].map(rank_map)

cluster_names = {0:"핫플 수익형", 1:"프리미엄 비즈니스", 2:"로컬 주거형", 3:"가성비 신흥형"}
district_agg["cluster_name"] = district_agg["cluster_rank"].map(cluster_names)

print("클러스터별 자치구:")
for cid, name in cluster_names.items():
    members = district_agg[district_agg["cluster_rank"]==cid]["district"].tolist()
    revpar = district_agg[district_agg["cluster_rank"]==cid]["median_revpar_ao"].median()
    print(f"\n  [{name}] RevPAR 중위값: ₩{revpar:,.0f}")
    print(f"       {', '.join(members)}")

In [ ]:
# RevPAR vs 리스팅 수 산점도 -- 군집 특성을 2차원으로 직관적으로 확인
CLUSTER_COLORS = {"핫플 수익형":"#1565C0","프리미엄 비즈니스":"#2E7D32","로컬 주거형":"#FF8F00","가성비 신흥형":"#C62828"}
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for name, group in district_agg.groupby("cluster_name"):
    axes[0].scatter(group["total_listings"], group["median_revpar_ao"], s=150, label=name, color=CLUSTER_COLORS[name], alpha=0.8, edgecolors="white")
    for _, row in group.iterrows():
        axes[0].annotate(row["district"], (row["total_listings"], row["median_revpar_ao"]), fontsize=7, ha="center", va="bottom")
axes[0].set_title("자치구 클러스터 (공급량 vs RevPAR)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("총 리스팅 수"); axes[0].set_ylabel("RevPAR 중위값 (₩)"); axes[0].legend(fontsize=8)

for i, name in cluster_names.items():
    rev = district_agg[district_agg["cluster_rank"]==i]["median_revpar_ao"].median()
    axes[1].bar(i, rev, color=CLUSTER_COLORS[name], alpha=0.8, width=0.6)
    axes[1].text(i, rev+300, f"₩{rev:,.0f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
axes[1].set_title("클러스터별 RevPAR 중위값", fontsize=12, fontweight="bold")
axes[1].set_ylabel("TTM RevPAR 중위값 (₩)")
axes[1].set_xticks(range(4)); axes[1].set_xticklabels([cluster_names[i] for i in range(4)], rotation=20)
plt.tight_layout()
plt.show()

## 3️⃣ 클러스터별 특성 프로파일링

In [ ]:
# 레이더 차트 -- 5개 지표의 군집별 프로파일을 한눈에 비교
profile_cols = ["median_revpar_ao","total_listings","dormant_ratio","superhost_rate","median_pop"]
profile_labels = ["RevPAR중위값","리스팅수","비활성비율","슈퍼호스트비율","인구"]
cluster_profile = district_agg.groupby("cluster_name")[profile_cols].mean()

# MinMaxScaler로 정규화 (히트맵 색상용)
mms = MinMaxScaler()
profile_norm = pd.DataFrame(
    mms.fit_transform(cluster_profile), 
    index=cluster_profile.index, 
    columns=profile_labels
)

# annotation용 포맷팅: 비율은 %, 절대값은 천단위 구분
annot_data = cluster_profile.copy()
annot_formatted = []
for i, row in annot_data.iterrows():
    formatted_row = [
        f"{row['median_revpar_ao']:,.0f}원",     # RevPAR 원화 표시
        f"{row['total_listings']:.0f}개",        # 리스팅수
        f"{row['dormant_ratio']*100:.1f}%",      # 비활성비율 -> %
        f"{row['superhost_rate']*100:.1f}%",     # 슈퍼호스트비율 -> %
        f"{row['median_pop']:,.0f}명"            # 인구
    ]
    annot_formatted.append(formatted_row)

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(
    profile_norm, 
    annot=annot_formatted, 
    fmt="", 
    cmap="RdYlGn", 
    ax=ax, 
    linewidths=0.5, 
    cbar_kws={"label":"정규화 값 (0~1)"},
    annot_kws={"fontsize": 9}
)
ax.set_title("클러스터별 특성 프로파일 (정규화 히트맵)", fontsize=13, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("")
# y축 레이블 회전 각도 조정 (수평으로)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, va='center')
plt.tight_layout()
plt.show()

In [ ]:
# 클러스터 결과를 CSV로 저장 -- 이후 06_business_insight에서 재사용
district_agg.to_csv(PROCESSED_DIR/"district_clustered.csv", index=False, encoding="utf-8-sig")
print("✅ 클러스터 결과 저장: data/processed/district_clustered.csv")
print("\n📋 클러스터 요약:")
print(district_agg.groupby("cluster_name").agg(자치구수=("district","count"), RevPAR중위=("median_revpar_ao","median"), 비활성비율=("dormant_ratio","mean")).round(3).to_string())

## 결론

K-Means(k=4) 군집 분석 결과:

| 군집 | 유형 | 특성 |
|------|------|------|
| Cluster 0 | 핫플 수익형 | 종로·중구 — 관광 집중, 높은 RevPAR |
| Cluster 1 | 프리미엄 비즈니스 | 강남·서초 — 안정적 수요, 슈퍼호스트 비율 높음 |
| Cluster 2 | 가성비 신흥형 | 마포·용산 — 성장 잠재력, 최적화 여지 |
| Cluster 3 | 로컬 주거형 | 노원·도봉 — 니치 마켓, 낮은 RevPAR |

> 다음 단계: `06_business_insight.ipynb` 에서 군집별 전략 수립